# Agents with Tools versus Tasks with Tools in CrewAI

### The Role of Tools: Agent-Centric versus Task-Centric

A key decision in CrewAI is *how* to give tools to your agents. There are two main strategies, and understanding the difference is crucial for building robust applications.

**1. Agent-Centric Approach (The Generalist):**
This is the most common method. You give an agent a 'toolbox' containing all the tools it might possibly need. The agent then uses its intelligence (the LLM's reasoning ability) to decide which tool is best for the situation at hand. This is flexible but can sometimes lead to the agent making mistakes or using tools inefficiently.


**2. Task-Centric Approach (The Specialist):**
In this more advanced approach, you don't give the tools to the agent directly. Instead, you attach specific tools to the specific **Tasks** that require them. When the agent starts a task, it is temporarily granted access to only the tools needed for that job and agent could decides whether it requires to call tool or not. This creates a much more focused and predictable workflow.



`For this lab, we need SERPER_API_KEY and OPENAI_API_KEY`

In [3]:
import warnings
warnings.filterwarnings('ignore') #Keeps Jupyter Notebook clean (not part of functionality)

from crewai import LLM, Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool, SerperDevTool

import litellm
litellm.ssl_verify = False


In [2]:
llm = LLM(model="openai/gpt-4o-mini")
web_search_tool = SerperDevTool()

## Create a custom tool

In [ ]:
from crewai.tools import tool
import re

@tool("Add Two Numbers Tool")
def add_numbers(data: str) -> int:
    """
    Extracts and adds integers from the input string.
    Example input: 'add 1 and 2' or '[1,2,3,4]'
    Output: sum of the numbers
    """
    # Find all integers in the string
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return sum(numbers)

## Creating our PDF Search Tool

In [4]:
pdf_search_tool = PDFSearchTool(
    pdf="The_Daily_Dish_FAQ.pdf", # "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf"
    config=dict(
        embedder=dict(
            provider="openai", # huggingface
            config=dict(
                model="text-embedding-3-small" # sentence-transformers/all-MiniLM-L6-v2
            )
        )
    )
)

## Approach 1: The Standard Method (Agent-Centric Tools)

### Create Agent, Task and Crew

In [5]:
agent_centric_agent = Agent(
    role="The Daily Dish Inquiry Specialist",
    goal="""Accurately answer customer questions about The Daily Dish restaurant. 
    You must decide whether to use the restaurant's FAQ PDF or a web search to find the best answer.""",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You have access to two tools: one for searching the restaurant's FAQ document and another for searching the web.
    Your job is to analyze the user's question and choose the most appropriate tool to find the information needed to provide a helpful response.""",
    tools=[pdf_search_tool, web_search_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [6]:
agent_centric_task = Task(
    description="Answer the following customer query: '{customer_query}'. "
                "Analyze the question and use the tools at your disposal (PDF search or web search) to find the most relevant information. "
                "Synthesize the findings into a clear and friendly response.",
    expected_output="A comprehensive and well-formatted answer to the customer's query.",
    agent=agent_centric_agent
)

In [7]:
agent_centric_crew = Crew(
    agents=[agent_centric_agent],
    tasks=[agent_centric_task],
    process=Process.sequential,
    verbose=False
)

In [ ]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_agent_centric = agent_centric_crew.kickoff(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_agent_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")

# QUESTIONS THAT COULD BE ASKED:
    # What are the timings?
    # What is the phone number?
    # What is the location?

## Approach 2: A More Focused Method (Task-Centric Tools)

### Create Agent, Tasks and Crew

In [9]:
task_centric_agent = Agent(
    role="Customer Service Specialist",
    goal="Provide exceptional customer service by following a multi-step process to answer customer questions accurately.",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You are an expert at following instructions. You will be given a sequence of tasks to complete.
    For each task, you will be provided with the specific tool needed to accomplish it.
    Your job is to execute each task diligently and pass the results to the next step.""",
    tools=[], # The agent is not given any tools directly
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [14]:
faq_pdf_search_task = Task(
    description="Search the restaurant's FAQ PDF for information related to the customer's query: '{customer_query}'.",
    expected_output="A snippet of the most relevant information from the PDF, or a statement that the information was not found.",
    tools=[pdf_search_tool], # Tool assigned directly to the task
    agent=task_centric_agent
)

faq_web_search_task = Task(
    description="Search the web only once if required, for information related to the customer's query: '{customer_query}'.",
    expected_output="A snippet of the most relevant information from the Web, or a statement that the information was not found.",
    tools=[web_search_tool], # Tool assigned directly to the task
    context=[faq_pdf_search_task],
    agent=task_centric_agent
)

response_drafting_task = Task(
    description="Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the customer's query: '{customer_query}'.",
    expected_output="The final, customer-facing response.",
    agent=task_centric_agent,
    context=[faq_pdf_search_task, faq_web_search_task]
)

In [15]:
task_centric_crew = Crew(
    agents=[task_centric_agent],
    tasks=[faq_pdf_search_task, faq_web_search_task, response_drafting_task],
    process=Process.sequential,
    verbose=True
)

In [16]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_task_centric = task_centric_crew.kickoff(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_task_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")

# QUESTIONS THAT COULD BE ASKED:
    # What are the timings?
    # What is the phone number?
    # What is the location?


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ded848ce-af13-4251-ba90-26b1c4dd2af0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the restaurant's FAQ PDF for information related to the customer's query: 'what are the           │
│  timings?'.                                                                                                     │
│  ID: c1875882-6a77-4c90-a926-b4e8a7d95b59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│  Task: Search the restaurant's FAQ PDF for information related to the customer's query: 'what are the           │
│  timings?'.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {'query': 'what are the timings?'}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the   │
│  customer's query: 'what are the timings?'.                                                                     │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- The Daily Dish Assistant ---
Thank you for your inquiry about our timings! The Daily Dish is open Monday to Friday from 11:00 AM to 10:00 PM. On Saturday and Sunday, we welcome you from 10:00 AM to 11:00 PM. We look forward to serving you soon!
--------------------------------


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ded848ce-af13-4251-ba90-26b1c4dd2af0                                                                       │
│  Final Output: Thank you for your inquiry about our timings! The Daily Dish is open Monday to Friday from       │
│  11:00 AM to 10:00 PM. On Saturday and Sunday, we welcome you from 10:00 AM to 11:00 PM. We look forward to     │
│  serving you soon!                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thank you for chatting. Have a great day!
